# InfraScan: Concrete Crack Classification (v3) - LOCAL LAPTOP edition (RTX 2050, 4 GB)
**Binary classification: Cracked (1) vs Uncracked (0)** | Dataset: Ozgenel / METU, 40,000 patches (227 x 227), downloaded from the Kaggle mirror.

**Before you run (one time, in a terminal):**
1. Install the **CUDA build** of PyTorch from pytorch.org, for example `pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124` (use the command the site gives you for your driver).
2. Plug in the charger and set the Windows power mode to *Best performance*; close browsers and games so the 4 GB of GPU memory is free.
3. Start Jupyter from the folder where you want outputs; they go to `./infrascan_crack_classification`.

**Laptop-specific settings in this edition**
- Settings sized for 4 GB VRAM: mixed precision, batch sizes 64 / 64 / 32, automatic halving if memory runs out.
- Three candidate models (ResNet-18, MobileNetV3-Large, EfficientNet-B0). ConvNeXt-Tiny is left out to keep training time reasonable.
- Stops with a clear message if CUDA is not found, instead of silently training on the CPU.
- Resumable: if you close the lid or stop the kernel, run the cells again and training continues from the last checkpoint.
- If data loading stalls, set `NUM_WORKERS = 0` in section 2.

**Read this limitation first.** The 40,000 patches come from only 458 photos. A random patch split puts near-identical patches of the same photo in train and test, so the scores on this dataset are optimistic (typically 99.5%+). Use the optional external test cell (a different dataset) for an honest generalisation estimate.

## 1. Environment

In [1]:
# Install the CUDA build of PyTorch ONCE in a terminal first (see the note at the top). Do NOT reinstall torch here:
# a CPU-only torch would replace your CUDA build.
%pip install -q kagglehub scikit-learn seaborn matplotlib pandas opencv-python-headless pillow tqdm onnx onnxruntime

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, sys, gc, json, time, shutil, random, copy
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, f1_score, roc_auc_score,
                             average_precision_score, precision_recall_curve,
                             classification_report)

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True   # faster convs; tiny run-to-run nondeterminism is acceptable here
seed_everything()

REQUIRE_GPU = True   # set False only for a CPU smoke test
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
VRAM_GB = 0.0
print("PyTorch:", torch.__version__, "| device:", device)
if device.type == "cuda":
    p = torch.cuda.get_device_properties(0)
    VRAM_GB = p.total_memory / 1024**3
    print(f"GPU: {p.name} | VRAM: {VRAM_GB:.1f} GB | compute capability {p.major}.{p.minor}")
else:
    print("WARNING: no CUDA GPU detected. Training on CPU will be very slow. "
          "On a local PC install the CUDA build of PyTorch; on Colab choose Runtime > Change runtime type > GPU.")
if REQUIRE_GPU and device.type != "cuda":
    raise RuntimeError("No CUDA GPU found. Install the CUDA build of PyTorch (pytorch.org) and update your NVIDIA driver, then restart the kernel.")

d:\project\aiml prime\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.11.0+cu128 | device: cuda
GPU: NVIDIA GeForce RTX 2050 | VRAM: 4.0 GB | compute capability 8.6


## 2. Configuration and storage

In [3]:
CFG = {
    "model_candidates": ["resnet18", "mobilenet_v3_large", "efficientnet_b0"],  # tuned for the 4 GB RTX 2050
    "epochs": 8,
    "patience": 3,            # early stopping on validation F1
    "lr": 3e-4,
    "weight_decay": 1e-2,
    "label_smoothing": 0.05,
    "img_size": 224,
    "val_frac": 0.15,
    "test_frac": 0.15,
    "target_recall": 0.99,    # threshold is tuned on the validation set to reach this crack recall
}
# Batch sizes that fit a 4 GB GPU (RTX 2050) with mixed precision. Scaled up automatically on bigger GPUs.
BATCH_4GB = {"resnet18": 64, "mobilenet_v3_large": 64, "efficientnet_b0": 32, "convnext_tiny": 16, "resnet50": 16}
BATCH_OVERRIDE = {}   # filled automatically if a model runs out of GPU memory

def batch_size_for(name):
    if name in BATCH_OVERRIDE:
        return BATCH_OVERRIDE[name]
    if device.type != "cuda":
        return 16
    scale = 1 if VRAM_GB <= 5 else (2 if VRAM_GB <= 12 else 4)
    return BATCH_4GB.get(name, 32) * scale

# If data loading ever stalls on your machine, set NUM_WORKERS = 0 (slower but always works).
NUM_WORKERS = min(4, os.cpu_count() or 2)   # v3: the Dataset is now importable, so Windows can use workers too

IS_COLAB = False
BASE_DIR = Path.cwd() / "infrascan_crack_classification"

CKPT_DIR, MODELS_DIR, OUT_DIR = BASE_DIR / "checkpoints", BASE_DIR / "models", BASE_DIR / "outputs"
for d in (CKPT_DIR, MODELS_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("Project folder:", BASE_DIR)
print("Num workers:", NUM_WORKERS)

Project folder: d:\project\aiml prime\project\infosys\collab\infrascan_crack_classification
Num workers: 4


## 3. Dataset: Concrete Crack Images for Classification
- **Author / source:** Cagla Firat Ozgenel, Middle East Technical University (METU), Mendeley Data, DOI `10.17632/5y9wdsg2zt.2` (<https://data.mendeley.com/datasets/5y9wdsg2zt/2>). Please cite Ozgenel & Gonenc Sorguc, ISARC 2018.
- **Size:** 40,000 RGB images of **227 x 227** px: 20,000 `Positive` (crack) + 20,000 `Negative` (no crack).
- **Origin:** cut from only **458** high-resolution (4032 x 3024) photos of METU campus buildings. No augmentation applied.
- **Download here:** Kaggle mirror `arunrk7/surface-crack-detection` through `kagglehub`. If that fails, download the zip from Mendeley, unzip, and set `DATASET_DIR_OVERRIDE`.

In [4]:
KAGGLE_SLUG = "arunrk7/surface-crack-detection"
MENDELEY_URL = "https://data.mendeley.com/datasets/5y9wdsg2zt/2"
DATASET_DIR_OVERRIDE = None     # e.g. r"D:\datasets\SurfaceCrack"  (folder that contains Positive/ and Negative/)
SMOKE_TEST = False              # True = tiny FAKE dataset only to test that the pipeline runs. Results are meaningless.
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}

def find_class_dirs(base):
    for root, dirs, _ in os.walk(base):
        low = {d.lower(): d for d in dirs}
        if "positive" in low and "negative" in low:
            return Path(root) / low["positive"], Path(root) / low["negative"]
    return None

def make_smoke_dataset(root, n=150):
    pos, neg = root / "Positive", root / "Negative"
    pos.mkdir(parents=True, exist_ok=True); neg.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(SEED)
    for i in range(n):
        base = cv2.GaussianBlur(rng.normal(160, 25, (227, 227)).clip(0, 255).astype(np.uint8), (3, 3), 0)
        img = cv2.cvtColor(base, cv2.COLOR_GRAY2BGR)
        cv2.imwrite(str(neg / f"{i:05d}.jpg"), img)
        x, y, pts = int(rng.integers(20, 200)), 0, []
        pts.append((x, y))
        while y < 227:
            x = int(np.clip(x + rng.integers(-15, 16), 10, 215)); y += int(rng.integers(15, 35)); pts.append((x, min(227, y)))
        for a, b in zip(pts[:-1], pts[1:]):
            cv2.line(img, a, b, (40, 40, 40), int(rng.integers(2, 4)))
        cv2.imwrite(str(pos / f"{i:05d}.jpg"), img)
    return pos, neg

def acquire_dataset():
    if SMOKE_TEST:
        print("!" * 70, "\nSMOKE TEST MODE: using a tiny FAKE dataset. Do NOT report these results.\n" + "!" * 70)
        return (*make_smoke_dataset(BASE_DIR / "smoke_dataset"), "synthetic smoke test")
    if DATASET_DIR_OVERRIDE:
        found = find_class_dirs(DATASET_DIR_OVERRIDE)
        if not found:
            raise FileNotFoundError(f"No Positive/Negative folders under {DATASET_DIR_OVERRIDE}")
        return (*found, f"local folder {DATASET_DIR_OVERRIDE}")
    try:
        import kagglehub
        path = kagglehub.dataset_download(KAGGLE_SLUG)
        found = find_class_dirs(path)
        if not found:
            raise FileNotFoundError("Downloaded, but Positive/Negative folders were not found")
        return (*found, f"kaggle:{KAGGLE_SLUG}")
    except Exception as e:
        raise RuntimeError(
            f"Dataset download failed ({e}).\n"
            f"Download it manually from {MENDELEY_URL} (or kaggle.com/datasets/{KAGGLE_SLUG}), unzip it, "
            f"set DATASET_DIR_OVERRIDE to the folder that contains Positive/ and Negative/, and re-run.") from e

POS_DIR, NEG_DIR, DATASET_SOURCE = acquire_dataset()
list_images = lambda d: sorted(p for p in Path(d).iterdir() if p.suffix.lower() in IMG_EXT)
pos_images, neg_images = list_images(POS_DIR), list_images(NEG_DIR)

print("Source:", DATASET_SOURCE)
print(f"Positive (cracked):   {len(pos_images):,}")
print(f"Negative (uncracked): {len(neg_images):,}")
print(f"Total:                {len(pos_images) + len(neg_images):,}")
assert len(pos_images) > 0 and len(neg_images) > 0, "Empty class folder"
if not SMOKE_TEST and (len(pos_images) != 20000 or len(neg_images) != 20000):
    print("WARNING: expected 20,000 images per class. Check that the download is complete.")

def _img_size(p):
    with Image.open(p) as im:
        return im.size
sizes = {_img_size(p) for p in random.sample(pos_images + neg_images, min(300, len(pos_images) + len(neg_images)))}
print("Image sizes in a random sample of 300:", sizes)

KeyboardInterrupt: 

## 4. Visual check

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6.5))
for col in range(5):
    for row, (pool, title, color) in enumerate([(pos_images, "Cracked", "darkred"), (neg_images, "Uncracked", "darkgreen")]):
        p = random.choice(pool)
        axes[row, col].imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
        axes[row, col].set_title(f"{title}\n{p.name}", fontsize=9, color=color)
        axes[row, col].axis("off")
plt.tight_layout(); plt.savefig(OUT_DIR / "sample_images.png", dpi=130); plt.show()

## 5. Stratified split (70 / 15 / 15)

In [ ]:
df_all = pd.DataFrame({"path": [str(p) for p in neg_images + pos_images],
                       "label": [0] * len(neg_images) + [1] * len(pos_images)})
hold = CFG["val_frac"] + CFG["test_frac"]
train_df, tmp_df = train_test_split(df_all, test_size=hold, stratify=df_all.label, random_state=SEED)
val_df, test_df = train_test_split(tmp_df, test_size=CFG["test_frac"] / hold, stratify=tmp_df.label, random_state=SEED)
for n, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{n:5s}: {len(d):6,}  class counts {d.label.value_counts().sort_index().to_dict()}")
pd.concat([train_df.assign(split="train"), val_df.assign(split="val"), test_df.assign(split="test")]).to_csv(OUT_DIR / "split.csv", index=False)

## 6. Augmentation and data loaders
Spatial + photometric augmentation to mimic field conditions (zoom, tilt, shadows, blur). No random erasing: it can erase the thin crack and create label noise.

In [ ]:
IMG = CFG["img_size"]
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG, scale=(0.7, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomApply([transforms.RandomRotation(20)], p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.02),
    transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, 1.5))], p=0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([transforms.Resize((IMG, IMG)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

# The Dataset lives in a real .py file so that Windows worker processes can import it
# (classes defined inside a notebook cell cannot be pickled by "spawn" workers).
DATA_MODULE = BASE_DIR / "infrascan_data.py"
DATA_MODULE.write_text('''from PIL import Image
from torch.utils.data import Dataset


class CrackDataset(Dataset):
    def __init__(self, paths, labels, tf):
        self.paths, self.labels, self.tf = list(paths), list(labels), tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            img = im.convert("RGB")
        return self.tf(img), self.labels[i], self.paths[i]
''')
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
import importlib, infrascan_data
importlib.reload(infrascan_data)
CrackDataset = infrascan_data.CrackDataset

def make_loader(df, tf, bs, train, persistent=None):
    ds = CrackDataset(df.path.tolist(), df.label.tolist(), tf)
    kw = dict(batch_size=bs, shuffle=train, drop_last=train, num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")
    if NUM_WORKERS > 0 and (train if persistent is None else persistent):
        kw.update(persistent_workers=True, prefetch_factor=4)
    elif NUM_WORKERS > 0:
        kw.update(prefetch_factor=4)
    return DataLoader(ds, **kw)


## 7. Candidate models (all ImageNet-pretrained, fully fine-tuned)
| Model | Params | Why it is here |
|---|---|---|
| ResNet-18 | 11.2 M | Project baseline; very robust |
| MobileNetV3-Large | 5.5 M | Fastest; best for edge/drone/phone deployment |
| EfficientNet-B0 | 5.3 M | Usually the best accuracy per parameter |

The notebook trains each one, then picks the winner by validation macro-F1 (ties within 0.001 go to the faster model).

In [ ]:
def build_model(name, pretrained=True):
    if name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, 2))
    elif name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, 2))
    elif name == "convnext_tiny":
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, 2)
    elif name == "mobilenet_v3_large":
        m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2 if pretrained else None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, 2)
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier[1].in_features, 2))
    else:
        raise ValueError(f"Unknown model {name}")
    return m

GRADCAM_LAYER = {
    "resnet18": lambda m: m.layer4[-1], "resnet50": lambda m: m.layer4[-1],
    "mobilenet_v3_large": lambda m: m.features[-1], "efficientnet_b0": lambda m: m.features[-1],
    "convnext_tiny": lambda m: m.features[-1],
}
count_params = lambda m: sum(p.numel() for p in m.parameters())
for n in CFG["model_candidates"]:
    print(f"{n:20s} params: {count_params(build_model(n, pretrained=False)) / 1e6:5.2f} M | batch size on this device: {batch_size_for(n)}")

## 8. Checkpointing, metrics and training engine (auto-resume)

In [ ]:
class CkptManager:
    def __init__(self, folder, name):
        self.latest, self.best = folder / f"{name}_latest.pth", folder / f"{name}_best.pth"
        self.hist = folder / f"{name}_history.json"
    def save(self, epoch, model, optimizer, scheduler, scaler, history, is_best):
        state = {"epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                 "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), "history": history}
        tmp = self.latest.with_suffix(".tmp")
        torch.save(state, tmp); os.replace(tmp, self.latest)          # atomic: no corrupt file if interrupted
        json.dump(history, open(self.hist, "w"), indent=2)
        if is_best:
            torch.save({"model": model.state_dict(), "epoch": epoch}, self.best)
    def load(self, model, optimizer, scheduler, scaler):
        if not self.latest.exists():
            return 0, new_history()
        ck = torch.load(self.latest, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
        scheduler.load_state_dict(ck["scheduler"]); scaler.load_state_dict(ck["scaler"])
        return ck["epoch"], ck["history"]

def new_history():
    return {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "val_f1", "val_auc", "epoch_sec"]}

def compute_metrics(y, p, thr=0.5):
    y, pred = np.asarray(y), (np.asarray(p) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "threshold": float(thr), "accuracy": float((tn + tp) / len(y)),
        "precision": float(tp / max(tp + fp, 1)), "recall": float(tp / max(tp + fn, 1)),
        "specificity": float(tn / max(tn + fp, 1)),
        "f1_crack": float(f1_score(y, pred, zero_division=0)),
        "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0)),
        "roc_auc": float(roc_auc_score(y, p)), "pr_auc": float(average_precision_score(y, p)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

def to_dev(x):
    return x.to(device, non_blocking=True).contiguous(memory_format=torch.channels_last)

def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    loss_sum, correct, n = torch.zeros((), device=device), torch.zeros((), device=device), 0
    for x, y, _ in tqdm(loader, desc="train", leave=False):
        x, y = to_dev(x), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            out = model(x); loss = criterion(out, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        loss_sum += loss.detach() * y.size(0); correct += (out.argmax(1) == y).sum(); n += y.size(0)
    return loss_sum.item() / n, correct.item() / n

@torch.no_grad()
def predict(model, loader):
    model.eval(); probs, labels, paths, loss_sum = [], [], [], 0.0
    for x, y, p in tqdm(loader, desc="eval", leave=False):
        x, yd = to_dev(x), y.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            out = model(x)
        out = out.float()
        loss_sum += F.cross_entropy(out, yd, reduction="sum").item()
        probs.append(F.softmax(out, 1)[:, 1].cpu().numpy()); labels.append(y.numpy()); paths += list(p)
    labels = np.concatenate(labels)
    return np.concatenate(probs), labels, paths, loss_sum / len(labels)

def train_model(name):
    print(f"\n{'=' * 70}\nTraining {name}\n{'=' * 70}")
    seed_everything()
    bs = batch_size_for(name)
    train_loader = make_loader(train_df, train_tf, bs, train=True)
    val_loader = make_loader(val_df, eval_tf, bs * 2, train=False, persistent=True)

    model = build_model(name).to(device).to(memory_format=torch.channels_last)
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    warm = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=1)
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(CFG["epochs"] - 1, 1), eta_min=1e-6)
    scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, [warm, cosine], milestones=[1])
    scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)

    ck = CkptManager(CKPT_DIR, name)
    start, hist = ck.load(model, optimizer, scheduler, scaler)
    best_f1 = max(hist["val_f1"]) if hist["val_f1"] else 0.0
    bad = len(hist["val_f1"]) - 1 - int(np.argmax(hist["val_f1"])) if hist["val_f1"] else 0
    if start:
        print(f"Resumed from epoch {start} (best val macro-F1 {best_f1:.4f})")

    for epoch in range(start + 1, CFG["epochs"] + 1):
        if bad >= CFG["patience"]:
            print(f"Early stopping: no improvement for {CFG['patience']} epochs."); break
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        p, y, _, v_loss = predict(model, val_loader)
        m = compute_metrics(y, p)
        scheduler.step()
        for k, v in [("train_loss", tr_loss), ("train_acc", tr_acc), ("val_loss", v_loss), ("val_acc", m["accuracy"]),
                     ("val_f1", m["macro_f1"]), ("val_auc", m["roc_auc"]), ("epoch_sec", time.time() - t0)]:
            hist[k].append(float(v))
        is_best = m["macro_f1"] > best_f1
        if is_best:
            best_f1, bad = m["macro_f1"], 0
        else:
            bad += 1
        print(f"[{name}] epoch {epoch:02d}/{CFG['epochs']} {time.time() - t0:5.0f}s | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
              f"val loss {v_loss:.4f} acc {m['accuracy']:.4f} F1 {m['macro_f1']:.4f} AUC {m['roc_auc']:.4f}" + ("  <- best" if is_best else ""))
        ck.save(epoch, model, optimizer, scheduler, scaler, hist, is_best)

    del model, optimizer, train_loader, val_loader
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return hist

## 9. Benchmark the candidates (resumable: if the session stops, re-run this cell)

In [ ]:
@torch.no_grad()
def latency_ms(model, bs=1, warmup=10, iters=50):
    model.eval()
    x = torch.randn(bs, 3, IMG, IMG, device=device).contiguous(memory_format=torch.channels_last)
    for _ in range(warmup):
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP): model(x)
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(iters):
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP): model(x)
    if device.type == "cuda": torch.cuda.synchronize()
    return (time.time() - t0) / iters * 1000

def train_with_oom_retry(name):
    """On a small GPU: if training runs out of memory, halve the batch size and resume from the last checkpoint."""
    while True:
        oom = False
        try:
            return train_model(name)
        except torch.cuda.OutOfMemoryError:
            oom = True
        if oom:                                   # handled outside the except block so the failed tensors can be freed
            gc.collect(); torch.cuda.empty_cache()
            cur = batch_size_for(name)
            if cur <= 8:
                raise RuntimeError(f"{name}: still out of GPU memory at batch size {cur}. Close other GPU programs or drop this model.")
            BATCH_OVERRIDE[name] = cur // 2
            print(f"CUDA out of memory for {name}: retrying with batch size {cur // 2}")

histories, rows = {}, []
for name in CFG["model_candidates"]:
    histories[name] = train_with_oom_retry(name)
    ck = CkptManager(CKPT_DIR, name)
    m = build_model(name, pretrained=False).to(device).to(memory_format=torch.channels_last)
    m.load_state_dict(torch.load(ck.best, map_location=device, weights_only=False)["model"])
    h = histories[name]
    rows.append({"model": name, "params_M": count_params(m) / 1e6,
                 "best_val_macro_f1": max(h["val_f1"]), "best_val_auc": h["val_auc"][int(np.argmax(h["val_f1"]))],
                 "epochs_run": len(h["val_f1"]), "avg_epoch_sec": float(np.mean(h["epoch_sec"])),
                 "latency_bs1_ms": latency_ms(m, 1), "latency_bs32_ms": latency_ms(m, 32)})
    del m
    if device.type == "cuda": torch.cuda.empty_cache()

bench = pd.DataFrame(rows).sort_values("best_val_macro_f1", ascending=False).reset_index(drop=True)
bench.to_csv(OUT_DIR / "model_benchmark.csv", index=False)
top = bench.best_val_macro_f1.max()
WINNER = bench[bench.best_val_macro_f1 >= top - 0.001].sort_values("latency_bs1_ms").iloc[0].model
print("\nBenchmark (validation set):")
display(bench.round(4))
print(f"\nSelected model: {WINNER}  (highest validation macro-F1; ties within 0.001 resolved by speed)")

## 10. Test-set evaluation of the selected model (with tuned threshold)

In [ ]:
best_ck = torch.load(CkptManager(CKPT_DIR, WINNER).best, map_location=device, weights_only=False)
model = build_model(WINNER, pretrained=False).to(device).to(memory_format=torch.channels_last)
model.load_state_dict(best_ck["model"]); model.eval()

bs = batch_size_for(WINNER) * 2
p_val, y_val, _, _ = predict(model, make_loader(val_df, eval_tf, bs, train=False))
p_test, y_test, test_paths, _ = predict(model, make_loader(test_df, eval_tf, bs, train=False))

# Threshold tuned on VALIDATION only: the highest-precision point that still reaches the target crack recall.
prec, rec, thr = precision_recall_curve(y_val, p_val)
ok = np.where(rec[:-1] >= CFG["target_recall"])[0]
THRESHOLD = float(thr[ok[np.argmax(prec[:-1][ok])]]) if len(ok) else 0.5
print(f"Tuned decision threshold (validation): {THRESHOLD:.4f}")

res = pd.DataFrame([{"setting": "threshold 0.50", **compute_metrics(y_test, p_test, 0.5)},
                    {"setting": f"tuned {THRESHOLD:.3f}", **compute_metrics(y_test, p_test, THRESHOLD)}])
res.insert(0, "model", WINNER)
res.to_csv(OUT_DIR / "test_report.csv", index=False)
display(res.round(4))
print(classification_report(y_test, (p_test >= THRESHOLD).astype(int), target_names=["Uncracked", "Cracked"], digits=4))

test_m = compute_metrics(y_test, p_test, THRESHOLD)
cm = confusion_matrix(y_test, (p_test >= THRESHOLD).astype(int), labels=[0, 1])
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
            xticklabels=["Uncracked", "Cracked"], yticklabels=["Uncracked", "Cracked"])
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"Test confusion matrix: {WINNER}")
plt.tight_layout(); plt.savefig(OUT_DIR / "confusion_matrix.png", dpi=130); plt.show()

# Training curves of the winner
h = histories[WINNER]; ep = range(1, len(h["train_loss"]) + 1)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].plot(ep, h["train_loss"], "o-", label="train"); ax[0].plot(ep, h["val_loss"], "s--", label="val"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(ep, h["val_acc"], "o-", color="green"); ax[1].set_title("Validation accuracy")
ax[2].plot(ep, h["val_f1"], "d-", color="purple"); ax[2].set_title("Validation macro-F1")
for a in ax: a.set_xlabel("epoch")
plt.tight_layout(); plt.savefig(OUT_DIR / "training_curves.png", dpi=130); plt.show()

## 11. Error analysis (what the model gets wrong)
Review these with a civil engineer: they show whether errors are hairline cracks, joints, stains or shadows.

In [ ]:
pred_test = (p_test >= THRESHOLD).astype(int)
wrong = np.where(pred_test != y_test)[0]
print(f"{len(wrong)} misclassified of {len(y_test)} test images (FN = missed cracks: {int(((pred_test == 0) & (y_test == 1)).sum())}, "
      f"FP = false alarms: {int(((pred_test == 1) & (y_test == 0)).sum())})")
if len(wrong):
    wrong = wrong[np.argsort(-np.abs(p_test[wrong] - 0.5))][:8]
    fig, axes = plt.subplots(2, 4, figsize=(14, 7)); axes = axes.ravel()
    for a in axes: a.axis("off")
    for a, i in zip(axes, wrong):
        a.imshow(Image.open(test_paths[i]).convert("RGB"))
        a.set_title(f"true {'crack' if y_test[i] else 'clean'} | P(crack)={p_test[i]:.2f}", fontsize=9, color="darkred")
    plt.tight_layout(); plt.savefig(OUT_DIR / "misclassified.png", dpi=130); plt.show()

## 12. Optional: external generalisation test (recommended)
Because the main dataset has only 458 source photos, test on a **different** dataset. Good choices: **NYA-Crack-Data** (Mendeley, 5,026 images, 227 x 227, folders `Crack` / `No Crack`, <https://data.mendeley.com/datasets/z93rb2m4fk/1>) or SDNET2018 (folders CD/UD, CP/UP, CW/UW are handled). Point `EXTERNAL_TEST_DIR` at a folder that has one sub-folder per class.

In [ ]:
EXTERNAL_TEST_DIR = None   # e.g. r"D:\datasets\NYA-Crack-Data" or the unzipped SDNET2018 folder

NEG_NAMES = {"negative", "no crack", "no_crack", "nocrack", "no-crack", "uncracked", "non-crack", "noncrack", "non_crack", "intact", "clean"}
POS_NAMES = {"positive", "crack", "cracked", "cracks"}
# SDNET2018 uses two-letter folders: C? = cracked, U? = uncracked (D = decks, P = pavements, W = walls)
NEG_NAMES |= {"ud", "up", "uw"}
POS_NAMES |= {"cd", "cp", "cw"}

if EXTERNAL_TEST_DIR:
    rows = []
    for root, dirs, files in os.walk(EXTERNAL_TEST_DIR):
        key = Path(root).name.lower()
        label = 0 if key in NEG_NAMES else (1 if key in POS_NAMES else None)
        if label is None: continue
        rows += [{"path": str(Path(root) / f), "label": label} for f in files if Path(f).suffix.lower() in IMG_EXT]
    ext_df = pd.DataFrame(rows)
    assert len(ext_df) and ext_df.label.nunique() == 2, "Could not find two recognisable class folders"
    print("External set:", len(ext_df), ext_df.label.value_counts().sort_index().to_dict())
    p_ext, y_ext, _, _ = predict(model, make_loader(ext_df, eval_tf, bs, train=False))
    ext_res = pd.DataFrame([{"setting": "threshold 0.50", **compute_metrics(y_ext, p_ext, 0.5)},
                            {"setting": f"tuned {THRESHOLD:.3f}", **compute_metrics(y_ext, p_ext, THRESHOLD)}])
    display(ext_res.round(4)); ext_res.to_csv(OUT_DIR / "external_test_report.csv", index=False)
    print("Expect a clear drop versus the in-dataset test score. That gap is the honest generalisation estimate.")
else:
    print("EXTERNAL_TEST_DIR not set: skipped.")

## 13. Explainability: Grad-CAM

In [ ]:
class GradCAM:
    """Gradient hook on the output tensor of the last conv block (avoids full_backward_hook / in-place ReLU problems)."""
    def __init__(self, model, layer):
        self.model, self.act, self.grad = model, None, None
        self.handle = layer.register_forward_hook(self._forward)
    def _forward(self, module, inp, out):
        self.act = out
        out.register_hook(lambda g: setattr(self, "grad", g))
    def __call__(self, x, class_idx=1):
        self.model.eval(); self.model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            out = self.model(x)
            out[0, class_idx].backward()
        w = self.grad.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.act).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0].detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, float(F.softmax(out.detach(), 1)[0, 1])
    def remove(self):
        self.handle.remove()

cam_model = copy.deepcopy(model).float().eval()          # fp32 copy for stable gradients
cam = GradCAM(cam_model, GRADCAM_LAYER[WINNER](cam_model))

pick = list(np.where((y_test == 1) & (pred_test == 1))[0][:3]) + list(np.where((y_test == 0) & (pred_test == 0))[0][:1])
fig, axes = plt.subplots(len(pick), 3, figsize=(11, 3.6 * len(pick)))
for r, i in enumerate(pick):
    img = Image.open(test_paths[i]).convert("RGB").resize((IMG, IMG))
    x = eval_tf(img).unsqueeze(0).to(device)
    heat, prob = cam(x)
    overlay = cv2.addWeighted(np.array(img), 0.6, cv2.cvtColor(cv2.applyColorMap(np.uint8(255 * heat), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB), 0.4, 0)
    for c, (im, t) in enumerate([(np.array(img), f"original ({'crack' if y_test[i] else 'clean'})"), (heat, "Grad-CAM"), (overlay, f"overlay, P(crack)={prob:.2f}")]):
        axes[r, c].imshow(im, cmap="jet" if c == 1 else None); axes[r, c].set_title(t, fontsize=9); axes[r, c].axis("off")
plt.tight_layout(); plt.savefig(OUT_DIR / "gradcam.png", dpi=130); plt.show()
cam.remove(); del cam_model

## 14. Export for the FastAPI backend (TorchScript + ONNX) with parity check

In [ ]:
export_model = copy.deepcopy(model).cpu().float().eval().to(memory_format=torch.contiguous_format)
dummy = torch.randn(1, 3, IMG, IMG)

weights_path = MODELS_DIR / f"infrascan_crack_{WINNER}_weights.pth"
ts_path = MODELS_DIR / f"infrascan_crack_{WINNER}_torchscript.pt"
onnx_path = MODELS_DIR / f"infrascan_crack_{WINNER}.onnx"

torch.save(export_model.state_dict(), weights_path)
torch.jit.trace(export_model, dummy).save(str(ts_path))

kw = dict(export_params=True, opset_version=17, do_constant_folding=True, input_names=["input_image"], output_names=["logits"],
          dynamic_axes={"input_image": {0: "batch"}, "logits": {0: "batch"}})
try:
    torch.onnx.export(export_model, dummy, str(onnx_path), dynamo=False, **kw)   # classic exporter: most stable
except TypeError:
    torch.onnx.export(export_model, dummy, str(onnx_path), **kw)                 # older PyTorch without the dynamo flag

import onnx, onnxruntime as ort
onnx.checker.check_model(onnx.load(str(onnx_path)))
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
x = torch.randn(4, 3, IMG, IMG)
with torch.no_grad():
    ref = export_model(x).numpy()
out = sess.run(None, {"input_image": x.numpy()})[0]
diff = float(np.abs(ref - out).max())
print(f"ONNX vs PyTorch max |logit difference|: {diff:.2e}  ({'OK' if diff < 1e-3 else 'CHECK THIS'})")

xb = {"input_image": np.random.randn(1, 3, IMG, IMG).astype(np.float32)}
for _ in range(5): sess.run(None, xb)
t0 = time.time()
for _ in range(30): sess.run(None, xb)
onnx_cpu_ms = (time.time() - t0) / 30 * 1000
print(f"ONNX Runtime CPU latency (batch 1): {onnx_cpu_ms:.1f} ms")

metadata = {
    "model_name": "InfraScan Concrete Crack Classifier", "version": "2.0.0", "architecture": WINNER,
    "input": {"size": [IMG, IMG], "channels": 3, "resize": "bilinear to 224x224, no crop", "mean": MEAN, "std": STD},
    "classes": ["Uncracked", "Cracked"], "decision_threshold": THRESHOLD, "threshold_selection": f"validation set, recall >= {CFG['target_recall']}",
    "dataset": {"name": "Concrete Crack Images for Classification (Ozgenel, METU)", "doi": "10.17632/5y9wdsg2zt.2",
                "source_used": DATASET_SOURCE, "images": int(len(df_all)), "source_photos": 458},
    "test_metrics_tuned_threshold": {k: round(v, 4) for k, v in test_m.items() if isinstance(v, float)},
    "onnx_cpu_latency_ms_bs1": round(onnx_cpu_ms, 1),
    "known_limitations": "Patches come from 458 photos of METU buildings; random split overestimates real-world accuracy. "
                         "Binary only: no crack width/length/severity. Validate on site imagery before relying on it.",
    "exported_at": datetime.now().isoformat(),
    "artifacts": {"weights": weights_path.name, "torchscript": ts_path.name, "onnx": onnx_path.name},
}
json.dump(metadata, open(MODELS_DIR / "model_metadata.json", "w"), indent=2)
print("Saved:", *[p.name for p in (weights_path, ts_path, onnx_path, MODELS_DIR / "model_metadata.json")], sep="\n  ")

## 15. Using the ONNX model in FastAPI
Copy the `.onnx` and `model_metadata.json` into your backend. The preprocessing must match training exactly (RGB, resize to 224 x 224, ImageNet mean/std).

```python
import json, numpy as np, onnxruntime as ort
from PIL import Image

class CrackClassifierService:
    def __init__(self, onnx_path, metadata_path):
        self.meta = json.load(open(metadata_path))
        self.session = ort.InferenceSession(onnx_path, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
        self.input_name = self.session.get_inputs()[0].name
        self.mean = np.array(self.meta["input"]["mean"], dtype=np.float32)
        self.std = np.array(self.meta["input"]["std"], dtype=np.float32)
        self.threshold = self.meta["decision_threshold"]

    def predict(self, image: Image.Image):
        img = image.convert("RGB").resize((224, 224), Image.BILINEAR)
        x = (np.asarray(img, dtype=np.float32) / 255.0 - self.mean) / self.std
        x = np.transpose(x, (2, 0, 1))[None]
        logits = self.session.run(None, {self.input_name: x})[0]
        e = np.exp(logits - logits.max(axis=1, keepdims=True))
        p = float((e / e.sum(axis=1, keepdims=True))[0, 1])
        return {"has_crack": p >= self.threshold, "crack_probability": p, "threshold": self.threshold}
```